In [ ]:
import pypsa
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

In [ ]:
TITLE_SIZE = 20

In [ ]:
# BAU
sub_folder = '5_8_BAU'
n_cp_reactive = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl0a_ch180a_ec_lv1.0_3h_E_mapped_TEP.nc')
n_hr_reactive = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl180a_ch180a_ec_lv1.0_3h_E_mapped_TEP.nc')
n_lr_reactive = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl7a_ch180a_ec_lv1.0_3h_E_mapped_TEP.nc')
n_lr_proactive = pypsa.Network(f"/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl7a_ch180a_ec_lvopt_3h_E_mapped_TEP.nc")
n_hr_proactive = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_c180a_ec_lvopt_3h_E.nc')

In [ ]:
# REM
sub_folder = '5_8_REM'

n_cp_reactive_rem = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl0a_ch180a_ec_lv1.0_REM-3h_E_mapped_TEP.nc')
n_hr_reactive_rem = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl180a_ch180a_ec_lv1.0_REM-3h_E_mapped_TEP.nc')
n_lr_reactive_rem = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl7a_ch180a_ec_lv1.0_REM-3h_E_mapped_TEP.nc')
n_lr_proactive_rem = pypsa.Network(f"/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_cl7a_ch180a_ec_lvopt_REM-3h_E_mapped_TEP.nc")
n_hr_proactive_rem = pypsa.Network(f'/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/{sub_folder}/elec_s210_c180a_ec_lvopt_REM-3h_E.nc')

In [ ]:
colors = n_hr_reactive.carriers.color

# Extract Statistics

In [ ]:
# Extract Line Statistics
# Function to mark lines as interregional or intraregional
def mark_interregional_lines(network):
    network.lines['bus0_zone'] = network.lines['bus0'].map(lambda x: network.buses.loc[x, 'reeds_zone'])
    network.lines['bus1_zone'] = network.lines['bus1'].map(lambda x: network.buses.loc[x, 'reeds_zone'])
    network.lines['interregional'] = network.lines['bus0_zone'] != network.lines['bus1_zone']
    return network
# Mark interregional lines for each network
n_hr_copperplate = mark_interregional_lines(n_cp_reactive)
n_hr_reactive = mark_interregional_lines(n_hr_reactive)
n_lr_reactive = mark_interregional_lines(n_lr_reactive)
n_lr_proactive = mark_interregional_lines(n_lr_proactive)
n_hr_proactive = mark_interregional_lines(n_hr_proactive)


# Calculate the proportion of interregional vs. intraregional transmission capacity
def calculate_transmission_proportions(network):

    # Calculate new capacity for each line (only positive values)
    new_capacity = (network.lines.s_nom_opt - network.lines.s_nom).clip(lower=0)  / 1e3  # Convert to GW
    new_capacity_length = network.lines.length * new_capacity

    # Calculate total new capacity
    total_new_capacity = new_capacity_length.sum()
    
    # Calculate interregional and intraregional capacity
    interregional_capacity = new_capacity_length[network.lines.interregional].sum()
    intraregional_capacity = new_capacity_length[~network.lines.interregional].sum()
    
    # Calculate new transmission CAPEX
    new_expenditure = new_capacity * network.lines.capital_cost * 1e3 / 1e6 # Convert back to MW then div for B$
    interregional_expenditure = new_expenditure[network.lines.interregional].sum()
    intraregional_expenditure = new_expenditure[~network.lines.interregional].sum()

    return {
        'total_capacity': total_new_capacity,
        'interregional_capacity': interregional_capacity,
        'intraregional_capacity': intraregional_capacity,
        'interregional_capacity_prop': interregional_capacity / total_new_capacity * 100,
        'intraregional_capacity_prop': intraregional_capacity / total_new_capacity * 100,
        'interregional_expenditure': interregional_expenditure,
        'intraregional_expenditure': intraregional_expenditure
    }

# Calculate proportions for each network
cp_reactive_data = calculate_transmission_proportions(n_cp_reactive)
lr_reactive_data = calculate_transmission_proportions(n_lr_reactive)
hr_reactive_data = calculate_transmission_proportions(n_hr_reactive)
lr_proactive_data = calculate_transmission_proportions(n_lr_proactive)
hr_proactive_data = calculate_transmission_proportions(n_hr_proactive)


In [ ]:
line_statistics_dict = {}
line_statistics_dict['Copperplate Reactive'] = cp_reactive_data
line_statistics_dict['LowRes Reactive'] = lr_reactive_data
line_statistics_dict['HighRes Reactive'] = hr_reactive_data
line_statistics_dict['LowRes Proactive'] = lr_proactive_data
line_statistics_dict['HighRes Proactive'] = hr_proactive_data

line_statistics_dict

In [ ]:
def extract_statistics(network):
    stats = network.statistics(nice_names=False)

    BESS_stats = stats.loc['StorageUnit'].sum(axis=0)
    idx = pd.MultiIndex.from_tuples([('StorageUnit', 'BESS')])
    bess_df = pd.DataFrame(BESS_stats).T
    bess_df.index = idx

    # Concatenate with original stats dataframe
    stats = pd.concat([stats, bess_df])

    # remove '4hr_battery_storage' from stats
    stats = stats.drop(index=[('StorageUnit', '4hr_battery_storage'), ('StorageUnit', 'battery')])

    return stats

# Create a dictionary to store statistics for each network
statistics_dict = {}

#assign network names to each network
n_cp_reactive.name = 'Copperplate Reactive'
n_lr_reactive.name = 'LowRes Reactive'
n_hr_reactive.name = 'HighRes Reactive'
n_lr_proactive.name = 'LowRes Proactive'
n_hr_proactive.name = 'HighRes Proactive'

# extract stats for REM networks
n_cp_reactive_rem.name = 'Copperplate Reactive REM'
n_lr_reactive_rem.name = 'LowRes Reactive REM'
n_hr_reactive_rem.name = 'HighRes Reactive REM'
n_lr_proactive_rem.name = 'LowRes Proactive REM'
n_hr_proactive_rem.name = 'HighRes Proactive REM'

# Iterate through each network
for network in [
    n_cp_reactive, n_lr_reactive, n_hr_reactive, n_lr_proactive, n_hr_proactive, 
    n_cp_reactive_rem, n_lr_reactive_rem, n_hr_reactive_rem, n_lr_proactive_rem, n_hr_proactive_rem
    ]:
    statistics_dict[network.name] = extract_statistics(network)

In [ ]:
capacities = [stats['Optimal Capacity'] for stats in statistics_dict.values()]
# only take Generators and StorageUnit from first level
capacities = [capacity.loc[['Generator', 'StorageUnit']] for capacity in capacities]
# drop the first level of the MultiIndex
capacities = [capacity.droplevel(0) for capacity in capacities]

In [ ]:
total_costs_dict = {}
for name, stats in statistics_dict.items():
    capex = stats['Capital Expenditure']
    capex = capex.loc[['Generator','Line', 'StorageUnit']]
    #combine 4hr_battery_storage and battery
    capex = capex.droplevel(0)
    capex.rename(index={'AC': 'Transmission'}, inplace=True)

    # do same for opex
    opex = stats['Operational Expenditure']
    opex = opex.loc[['Generator','Line', 'StorageUnit']] 
    opex = opex.droplevel(0)
    opex.rename(index={'AC': 'Transmission'}, inplace=True)

    # sum the values with same index
    total_costs = capex + opex

    # assign to new dictionary
    total_costs_dict[name] = total_costs


total_costs_dict[name]

In [ ]:
# Colors
colors = n_hr_reactive.carriers.color
colors['Transmission'] = colors['AC']
colors['BESS'] = colors['battery']
color_map = colors.to_dict()

# Total System Cost

In [ ]:
# Create stacked bar plot of total costs
total_costs_dict

LABEL_FONT_SIZE = 10
TITLE_FONT_SIZE = 15

# Extract data for plotting
network_names = list(total_costs_dict.keys())
technologies = list(total_costs_dict[network_names[0]].index)

# define order of networks
network_names = ['Copperplate Reactive', 'LowRes Reactive', 'HighRes Reactive', 'LowRes Proactive', 'HighRes Proactive']
network_names_rem = ['Copperplate Reactive REM', 'LowRes Reactive REM', 'HighRes Reactive REM', 'LowRes Proactive REM', 'HighRes Proactive REM']
# Define alternate names for x-axis labels
network_labels = {
    'Copperplate Reactive': 'CP-R',
    'LowRes Reactive': 'LR-R',
    'HighRes Reactive': 'HR-R', 
    'LowRes Proactive': 'LR-P',
    'HighRes Proactive': 'HR-P'
}

# Add labels to plot
network_rem_labels = {
    'Copperplate Reactive REM': 'CP-R',
    'LowRes Reactive REM': 'LR-R',
    'HighRes Reactive REM': 'HR-R',
    'LowRes Proactive REM': 'LR-P', 
    'HighRes Proactive REM': 'HR-P'
}


# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8,6))

# Add titles to subplots
ax1.set_title('BAU', fontsize=TITLE_FONT_SIZE)
ax2.set_title('Emissions Limit', fontsize=TITLE_FONT_SIZE)

# First subplot - original networks
x1 = np.arange(len(network_names))
width = 0.7
bottom1 = np.zeros(len(network_names))

# Second subplot - REM networks  
x2 = np.arange(len(network_names_rem))
bottom2 = np.zeros(len(network_names_rem))

# Create stacked bars for each subplot
for tech in technologies:
    # Values for original networks
    values1 = [total_costs_dict[network].loc[tech][2040] /1e9 for network in network_names]
    if not np.isnan(values1).all():
        ax1.bar(x1, values1, width, bottom=bottom1, label=tech, color=color_map[tech])
        bottom1 += np.nan_to_num(values1)
        
    # Values for REM networks    
    values2 = [total_costs_dict[network].loc[tech][2040] /1e9 for network in network_names_rem]
    if not np.isnan(values2).all():
        ax2.bar(x2, values2, width, bottom=bottom2, label=tech, color=color_map[tech])
        bottom2 += np.nan_to_num(values2)


# Add value labels for first subplot
for i, network in enumerate(network_names):
    bottom = 0
    for tech in technologies:
        value = total_costs_dict[network].loc[tech][2040] / 1e9
        if not np.isnan(value) and abs(value) >= 1.5:
            height = value
            ax1.text(i, bottom + height/2, f'{value:.1f}', 
                    ha='center', va='center',
                    color='black', 
                    fontsize=LABEL_FONT_SIZE)
            bottom += height
        elif not np.isnan(value):
            bottom += value
            
# Add value labels for second subplot            
for i, network in enumerate(network_names_rem):
    bottom = 0
    for tech in technologies:
        value = total_costs_dict[network].loc[tech][2040] / 1e9
        if not np.isnan(value) and abs(value) >= 1.5:
            height = value
            if tech == 'CCGT-95CCS':
                TEXT_COLOR = 'white'
            else:
                TEXT_COLOR = 'black'
            ax2.text(i, bottom + height/2, f'{value:.1f}',
                    ha='center', va='center',
                    color=TEXT_COLOR,
                    fontsize=LABEL_FONT_SIZE)
            bottom += height
        elif not np.isnan(value):
            bottom += value

# Calculate totals for first subplot
total_heights1 = []
for network in network_names:
    total = 0
    for tech in technologies:
        value = total_costs_dict[network].loc[tech][2040] / 1e9
        if not np.isnan(value):
            total += value
    total_heights1.append(total)

# Calculate totals for second subplot    
total_heights2 = []
for network in network_names_rem:
    total = 0
    for tech in technologies:
        value = total_costs_dict[network].loc[tech][2040] / 1e9
        if not np.isnan(value):
            total += value
    total_heights2.append(total)

# Add total and reduction labels for first subplot
ax1.text(0, total_heights1[0] + 0.05, f'{total_heights1[0]:.1f}',
         ha='center', va='bottom',
         color='black',
         fontsize=LABEL_FONT_SIZE)

for i in range(1, len(network_names)):
    pct_reduction = ((total_heights1[0] - total_heights1[i]) / total_heights1[0]) * 100
    ax1.text(i, total_heights1[i] + 0.05, f'-{pct_reduction:.1f}%',
             ha='center', va='bottom', 
             color='black',
             fontsize=LABEL_FONT_SIZE)

# Add total and reduction labels for second subplot             
ax2.text(0, total_heights2[0] + 0.05, f'{total_heights2[0]:.1f}',
         ha='center', va='bottom',
         color='black', 
         fontsize=LABEL_FONT_SIZE)

for i in range(1, len(network_names_rem)):
    pct_reduction = ((total_heights2[0] - total_heights2[i]) / total_heights2[0]) * 100
    ax2.text(i, total_heights2[i] + 0.05, f'-{pct_reduction:.1f}%',
             ha='center', va='bottom',
             color='black',
             fontsize=LABEL_FONT_SIZE)


# Set x-axis labels
ax1.set_xticks(x1)
ax1.set_xticklabels([network_labels[n] for n in network_names])
ax2.set_xticks(x2) 
ax2.set_xticklabels([network_labels[n.replace(' REM','')] for n in network_names_rem])


# Add legend to right of second subplot
ax2.legend(bbox_to_anchor=(1.05, 0.5), loc='center left', title='Technologies')

# Set same y limits for both plots
max_height = max(max(total_heights1), max(total_heights2))
ax1.set_ylim(0, max_height * 1.1)
ax2.set_ylim(0, max_height * 1.1)

# set one y axis label
ax1.set_ylabel('System Cost [B$/y]', fontsize=TITLE_FONT_SIZE)
#remove y axis tick labels for second subplot
ax2.set_yticklabels([])
plt.tight_layout()

## Analyze price divergence in final networks

In [ ]:
networks_bau = []
# networks_bau.add(n_cp_reactive)
networks_bau.append(n_lr_reactive)
networks_bau.append(n_hr_reactive)
networks_bau.append(n_lr_proactive)
networks_bau.append(n_hr_proactive)



In [ ]:
n_hr_reactive.buses_t.marginal_price